# Polynomial Regression 실험 — Poly + Ridge + 스태킹

## 실험 목표
트리 모델의 한계(RMSE ~1.37)를 돌파하기 위해 Polynomial Regression으로 전환

## 실험 흐름
1. **Poly degree 비교** (deg 1/2/3) → 2차: RMSE 1.69, 3차: RMSE 0.92
2. **Ridge 규제 + KFold** → RMSE 0.54
3. **Feature pruning** (base + TOP5) → RMSE 0.537
4. **Degree 앙상블** (2+3 혼합) → 스태킹 시작
5. **타겟 변환 다양화 스태킹** → RMSE 0.318 (핵심 돌파)
6. **scipy 가중치 최적화** → RMSE 0.309

## 성능 요약
| 단계 | RMSE | 핵심 기법 |
|------|------|----------|
| Poly(deg 1) | 7.26 | 23 features |
| Poly(deg 2) | 1.69 | 299 features |
| Poly(deg 3) | 0.92 | 2599 features |
| + Ridge + KFold | 0.54 | alpha=0.01 |
| + 타겟 변환 스태킹 | 0.318 | log/sqrt/YJ/raw |
| + 가중치 최적화 | **0.309** | scipy optimize |

---
## 1. 라이브러리 임포트 및 데이터 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random, os, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import (
    OneHotEncoder, PolynomialFeatures, StandardScaler, 
    RobustScaler, PowerTransformer
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, RidgeCV, ElasticNet
)
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

RANDOM_STATE = 42
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
# 데이터 로드
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
submission = pd.read_csv('sample_submission.csv')

print(f"Train: {train.shape}, Test: {test.shape}")

---
## 2. 피처 엔지니어링 (23개 파생변수)

EDA 결과를 바탕으로 칼로리 소모 관련 물리적 의미가 있는 파생변수를 생성함
- **기본 변환**: Height 통합, Temp_diff(체온편차), Duration 구간화
- **2차 상호작용**: Duration×BPM, Duration×TempDiff, BPM×TempDiff
- **비선형**: 제곱항, 3변수 곱, 비율
- **로그 변환**: Log_BPM, Log_Duration, Log_Weight_BPM_Dur
- **BMI**: 703 × Weight / Height²

In [ ]:
def create_features_safe(df):
    df = df.copy()
    
    # --- 기본 변환 ---
    df['Height_Total_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']  # 키 통합 (인치)
    df['Temp_diff'] = df['Body_Temperature(F)'] - 98.6  # 정상 체온 대비 편차 (운동 강도 반영)
    df['Duration_bin'] = pd.cut(  # 운동 시간 구간화
        df['Exercise_Duration'],
        bins=[-np.inf, 10, 20, 30, np.inf],
        labels=[0, 1, 2, 3]
    ).astype(int)
    
    # --- 2차 상호작용 ---
    df['Duration_x_BPM'] = df['Exercise_Duration'] * df['BPM']  # 운동 강도
    df['Duration_x_TempDiff'] = df['Exercise_Duration'] * df['Temp_diff']  # 운동 × 체온 상승
    df['BPM_x_TempDiff'] = df['BPM'] * df['Temp_diff']  # 심박수 × 체온 (심혈관 부하)
    
    # 핵심 변수: 물리식 기반
    df['Intensity'] = df['Duration_x_BPM']  # = Duration × BPM
    df['Effort'] = df['Weight(lb)'] * df['Intensity']  # = Weight × Duration × BPM
    
    # --- 비선형 ---
    df['Duration_sq'] = df['Exercise_Duration'] ** 2  # 시간 제곱 (곡선 관계 포착)
    df['Temp_diff_sq'] = df['Temp_diff'] ** 2  # 체온편차 제곱
    df['Dur_BPM_TempDiff'] = df['Exercise_Duration'] * df['BPM'] * df['Temp_diff']  # 3변수 곱
    
    # --- 비율 ---
    df['BPM_per_Duration'] = df['BPM'] / (df['Exercise_Duration'] + 1)  # 단위시간당 심박수
    df['TempDiff_per_Duration'] = df['Temp_diff'] / (df['Exercise_Duration'] + 1)  # 단위시간당 체온변화
    
    # --- BMI ---
    h2 = (df['Height_Total_Inches'] ** 2).replace(0, np.nan)
    df['BMI'] = 703 * df['Weight(lb)'] / h2  # BMI 공식 (파운드/인치 단위)
    df['BMI'] = df['BMI'].fillna(df['BMI'].median())
    
    # --- Weight 관련 ---
    df['Weight_x_Duration'] = df['Weight(lb)'] * df['Exercise_Duration']  # 체중 × 시간
    
    # --- 로그 변환 (큰 값 안정화) ---
    df['Log_BPM'] = np.log1p(df['BPM'])
    df['Log_Duration'] = np.log1p(df['Exercise_Duration'])
    df['Log_Weight_BPM_Dur'] = np.log1p(df['Weight(lb)'] * df['BPM'] * df['Exercise_Duration'])
    
    return df

---
## 3. 전처리 (OneHot 인코딩 + 파생변수)

In [ ]:
# 원본 분리
train_x_raw = train.drop(['ID', 'Calories_Burned'], axis=1).copy()
train_y = train['Calories_Burned'].astype(float).copy()
test_x_raw = test.drop(['ID'], axis=1).copy()

# 파생변수 생성
before_cols = list(train_x_raw.columns)
train_x = create_features_safe(train_x_raw)
test_x = create_features_safe(test_x_raw)

generated_cols = [c for c in train_x.columns if c not in before_cols]  # 파생변수 목록
print(f"파생변수 {len(generated_cols)}개 생성: {generated_cols}")

# OneHotEncoder: train에서만 fit → test는 transform만 (누수 방지)
cat_cols = [c for c in ['Gender', 'Weight_Status'] if c in train_x.columns]
if cat_cols:
    enc = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    enc.fit(train_x[cat_cols])  # train에서만 fit
    
    tr_cat = enc.transform(train_x[cat_cols])
    te_cat = enc.transform(test_x[cat_cols])
    cat_names = enc.get_feature_names_out(cat_cols)
    
    train_x = train_x.drop(columns=cat_cols)
    test_x = test_x.drop(columns=cat_cols)
    train_x[cat_names] = tr_cat
    test_x[cat_names] = te_cat

print(f"\n최종 피처 수: {train_x.shape[1]}")

---
## 4. 실험 — Polynomial Degree 비교 (1 / 2 / 3)

Polynomial Regression의 핵심: 피처 간 곱(상호작용)을 자동 생성
- degree=1: 원본 그대로 (23 features)
- degree=2: 모든 2차 조합 (299 features)  
- degree=3: 모든 3차 조합 (2599 features)

Calories = f(Duration × BPM × Temp) 같은 곱셈 구조를 포착하는 데 핵심

In [ ]:
# Polynomial Degree별 비교 실험
# train/val split으로 빠르게 확인
X_train, X_val, y_train, y_val = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)

for degree in [1, 2, 3]:
    # Polynomial → StandardScaler → Ridge 파이프라인
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    sc = StandardScaler()
    
    X_tr_poly = sc.fit_transform(poly.fit_transform(X_train))
    X_va_poly = sc.transform(poly.transform(X_val))
    
    model = Ridge(alpha=0.01, random_state=42)
    model.fit(X_tr_poly, y_train)
    pred = model.predict(X_va_poly)
    
    score = rmse(y_val, pred)
    n_feats = X_tr_poly.shape[1]
    print(f"Degree {degree}: RMSE={score:.4f} | features={n_feats}")

### 결과
- Degree 1 → 2: **-76%** 개선 (상호작용 항 추가 효과)
- Degree 2 → 3: **-46%** 개선 (3차 곱셈 구조 포착)
- Degree 3이 압도적으로 좋지만 피처 수 2599개 → 과적합 위험 → Ridge 규제 필수

---
## 5. Ridge 규제 + KFold 교차검증

- Ridge(L2 규제)로 과적합 방지: alpha가 클수록 규제 강함
- KFold(5-fold)로 일반화 성능을 OOF RMSE로 정확히 측정
- alpha를 logspace로 탐색 (1e-5 ~ 10, 25개 포인트)

In [ ]:
# Ridge alpha 탐색 + 5-Fold OOF
train_y_vals = train_y.values.astype(float)
alphas = np.logspace(-5, 1, 25)  # 1e-5 ~ 10까지 25개 로그 스케일

best_alpha, best_rmse_score = None, 1e18

for alpha in alphas:
    oof_preds = np.zeros(len(train_x))  # OOF 예측 저장 배열
    
    for tr_idx, va_idx in kf.split(train_x):
        # 각 fold 내부에서 poly/scaler fit → 누수 방지
        poly = PolynomialFeatures(degree=3, include_bias=False)
        sc = StandardScaler()
        
        X_tr = sc.fit_transform(poly.fit_transform(train_x.iloc[tr_idx]))  # train fold만 fit
        X_va = sc.transform(poly.transform(train_x.iloc[va_idx]))  # val fold는 transform만
        
        model = Ridge(alpha=alpha, random_state=42)
        model.fit(X_tr, train_y_vals[tr_idx])
        oof_preds[va_idx] = model.predict(X_va)
    
    score = rmse(train_y_vals, oof_preds)
    if score < best_rmse_score:
        best_rmse_score = score
        best_alpha = alpha

print(f"최적 alpha: {best_alpha:.6f}")
print(f"최적 OOF RMSE: {best_rmse_score:.6f}")

---
## 6. Feature Pruning — base + TOP5 파생변수만 유지

23개 파생변수 전부 사용하면 Polynomial 조합이 폭발적으로 증가
→ OOF 기반 제거 실험으로 핵심 5개만 선별 (나머지는 노이즈)

**TOP5**: Log_Weight_BPM_Dur, Effort, Weight_x_Duration, Log_Duration, Intensity
- 공통점: 모두 "곱 구조" 또는 물리식 기반 피처

In [ ]:
# Feature Pruning: base(원본+원핫) + TOP5만 유지
TOP5_FIXED = ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity']

base_feats = [c for c in train_x.columns if c not in generated_cols]  # 원본+원핫
keep_cols = base_feats + TOP5_FIXED

train_red = train_x[keep_cols].copy()
test_red = test_x[keep_cols].copy()

print(f"전체 피처: {train_x.shape[1]} → Pruning 후: {train_red.shape[1]}")
print(f"Base: {len(base_feats)} + TOP5: {len(TOP5_FIXED)}")

# Pruning 후 OOF RMSE 확인
oof_pruned = np.zeros(len(train_red))
for tr_idx, va_idx in kf.split(train_red):
    poly = PolynomialFeatures(degree=3, include_bias=False)
    sc = StandardScaler()
    X_tr = sc.fit_transform(poly.fit_transform(train_red.iloc[tr_idx]))
    X_va = sc.transform(poly.transform(train_red.iloc[va_idx]))
    
    model = Ridge(alpha=best_alpha, random_state=42)
    model.fit(X_tr, train_y_vals[tr_idx])
    oof_pruned[va_idx] = model.predict(X_va)

print(f"\nPruning 후 OOF RMSE: {rmse(train_y_vals, oof_pruned):.6f}")

---
## 7. 타겟 변환 다양화 스태킹 (아주 큰 개선!!)

**아이디어**: 동일한 Ridge+Poly 모델을 다양한 타겟 변환으로 학습
→ 각각의 예측이 다른 특성을 가짐 → 메타 모델이 최적 결합

**베이스 모델들 (동일 모델 × 다양한 타겟)**:
- identity: 원단위 y 그대로 학습
- log: log1p(y) 학습 → expm1 복원
- sqrt: sqrt(y) 학습 → 제곱 복원
- Yeo-Johnson: PowerTransformer 적용

**메타 모델**: RidgeCV (베이스 예측들을 입력으로 최적 결합 학습)

이 기법으로 **RMSE 0.58 → 0.318** (약 45% 개선)

In [ ]:
# 타겟 변환 다양화 스태킹
from sklearn.preprocessing import PowerTransformer

# 타겟 변환 정의
y = train_y.values.astype(float)
transforms = {
    'identity': (y, lambda p: p),                                    # 원단위
    'log':      (np.log1p(y), lambda p: np.expm1(p)),               # 로그 변환
    'sqrt':     (np.sqrt(np.clip(y, 0, None)), lambda p: np.clip(p, 0, None)**2),  # 제곱근
}

# Yeo-Johnson 변환
pt = PowerTransformer(method='yeo-johnson')
y_yj = pt.fit_transform(y.reshape(-1, 1)).ravel()
transforms['yeo-johnson'] = (y_yj, lambda p: pt.inverse_transform(p.reshape(-1, 1)).ravel())

# 각 변환별 Degree 2, 3 OOF 생성 (총 8개 베이스 모델)
DEGREE_LIST = [2, 3]
ALPHA = 0.01

oof_bases = {}  # {(transform, degree): oof_array}
test_bases = {}

for tname, (y_t, inv_fn) in transforms.items():
    for deg in DEGREE_LIST:
        key = f"{tname}_d{deg}"
        oof = np.zeros(len(train_red))
        test_pred = np.zeros(len(test_red))
        
        for tr_idx, va_idx in kf.split(train_red):
            poly = PolynomialFeatures(degree=deg, include_bias=False)
            sc = StandardScaler()
            
            X_tr = sc.fit_transform(poly.fit_transform(train_red.iloc[tr_idx]))
            X_va = sc.transform(poly.transform(train_red.iloc[va_idx]))
            X_te = sc.transform(poly.transform(test_red))
            
            model = Ridge(alpha=ALPHA, random_state=42)
            model.fit(X_tr, y_t[tr_idx])
            
            # 예측 후 역변환 → 원단위로 복원
            oof[va_idx] = inv_fn(model.predict(X_va))
            test_pred += inv_fn(model.predict(X_te)) / N_SPLITS
        
        oof_bases[key] = np.clip(oof, 0, None)
        test_bases[key] = np.clip(test_pred, 0, None)
        print(f"{key:20s} | OOF RMSE: {rmse(y, oof_bases[key]):.6f}")

# 메타 모델: RidgeCV
oof_stack = np.column_stack([oof_bases[k] for k in sorted(oof_bases)])
test_stack = np.column_stack([test_bases[k] for k in sorted(test_bases)])

meta = RidgeCV(alphas=np.logspace(-4, 2, 50))
meta.fit(oof_stack, y)

meta_oof = meta.predict(oof_stack)
meta_test = meta.predict(test_stack)

print(f"\n스태킹 메타 모델 OOF RMSE: {rmse(y, meta_oof):.6f}")
print(f"선택된 alpha: {meta.alpha_:.6f}")

---
## 8. scipy 가중치 최적화

스태킹 대신, 각 베이스 모델 예측의 최적 가중치를 직접 탐색
- 제약 조건: 가중치 합 = 1, 각 가중치 ≥ 0 (non-negative)
- scipy.optimize.minimize 사용

In [ ]:
# scipy 가중치 최적화
keys = sorted(oof_bases.keys())
oof_matrix = np.column_stack([oof_bases[k] for k in keys])
test_matrix = np.column_stack([test_bases[k] for k in keys])

def objective(weights):
    pred = np.clip(oof_matrix @ weights, 0, None)
    return rmse(y, pred)

# 초기값: 균등 가중치
n = len(keys)
x0 = np.ones(n) / n
bounds = [(0, 1)] * n
constraints = {'type': 'eq', 'fun': lambda w: w.sum() - 1}  # 가중치 합 = 1

result = minimize(objective, x0, bounds=bounds, constraints=constraints, method='SLSQP')

# 결과 출력
print("최적 가중치:")
for k, w in zip(keys, result.x):
    if w > 0.001:  # 의미 있는 가중치만 표시
        print(f"  {k:20s}: {w:.4f}")

final_oof = np.clip(oof_matrix @ result.x, 0, None)
final_test = np.clip(test_matrix @ result.x, 0, None)
print(f"\n최종 OOF RMSE: {rmse(y, final_oof):.6f}")

---
## 9. 제출 파일 생성

In [ ]:
submission['Calories_Burned'] = final_test
submission.to_csv('submit_poly_stacking.csv', index=False)
print("저장 완료: submit_poly_stacking.csv")
print(f"예측 통계: mean={final_test.mean():.2f}, min={final_test.min():.2f}, max={final_test.max():.2f}")

---
## 결론

| 실험 | RMSE | 핵심 |
|------|------|------|
| Poly(deg 3) + 전체 피처 | 0.92 | 모델 전환 |
| + Ridge + KFold | ~0.54 | 규제 + 안정화 |
| + Feature pruning | ~0.537 | 노이즈 제거 |
| + 타겟 변환 스태킹 | 0.318 | **핵심 돌파** |
| + scipy 가중치 최적화 | **0.309** | 미세 조정 |

**핵심 인사이트**: 
- 같은 모델이라도 타겟 변환을 다양화하면 서로 다른 예측 특성을 가짐
- 메타 모델이 이를 최적으로 결합하면 대폭 개선 가능
- Feature pruning은 Polynomial에서 특히 효과적 (조합 폭발 방지)